In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

# Two traps, both already hit once:
#   * Kaggle auto-extracts an uploaded .zip into a folder beside it, so
#     counting pyproject.toml files finds two roots.
#   * This kernel also mounts the precompute kernel's OUTPUT, which
#     contains its own stale copy of the project tree. Picking "the
#     shallowest match" would silently run last version's code.
# So: search only inside the code dataset, and verify afterwards.
inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))
candidates = [p for p in inputs.rglob("src/train/trainer.py") if "nanowm-code" in str(p)]
if not candidates:
    raise RuntimeError(f"no nanowm-code tree under {inputs}")
mounted = sorted(candidates, key=lambda q: len(q.parts))[0].parent.parent.parent
print("mounted project root:", mounted)

root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
assert (root / "scripts" / "run_m3_profile.py").exists(), sorted(p.name for p in root.iterdir())
# The KeyError('trainer') that killed the last attempt lived in this block.
import yaml
assert "trainer" in yaml.safe_load((root / "configs/m3_profile.yaml").read_text())
print("project root:", root)

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# CLAUDE.md: "Latents als Kaggle-Dataset ablegen, nicht neu berechnen."
# The 160-trajectory M3 tier WITH DINOv2 features was rendered once by the
# nanowm-m3-preflight kernel (274.9 MB, 21.6 min of weekly quota) and is
# published as the dataset says43/nanowm-m3-latents. Every M3/M4/M5 kernel
# mounts it instead of re-rendering.
import numpy as np, json

matches = [p for p in Path("/kaggle/input").rglob("manifest.json") if "nanowm-m3-latents" in str(p)]
if len(matches) != 1:
    raise RuntimeError(f"expected exactly one precomputed m3 manifest, got {matches}")
DATA_DIR = matches[0].parent
manifest = json.load(open(matches[0]))
print("data dir:", DATA_DIR)
print("trajectories:", manifest["num_trajectories"], "frames:", manifest["total_frames"])
print("dinov2:", manifest["dinov2_model_id"], manifest["dinov2_feature_dim"])

sample = sorted(DATA_DIR.glob("*.npz"))[0]
with np.load(sample) as data:
    print(sample.name, {k: data[k].shape for k in data.files})
    assert data["dinov2"].shape == (data["latents"].shape[0], 16, 384)

In [ ]:
# configs/m3_profile.yaml ships with ticket_hours: null and
# target_seconds_per_arm: null (tests/test_m3_profile.py guards both).
# The approved values are patched in only on this Kaggle copy.
#
# Ticket M3-A, approved 2026-09-05: 0.35 GPU-hours.
#   Phase 1: 4 arms x (30 warmup + 100 timed) steps at 15M
#   Phase 2: 3 Muon LRs + 1 AdamW reference x 600 steps
# ~15 min of real compute; the rest is margin for eight torch.compile
# invocations, whose cost the step timings deliberately exclude.
#
# The previous attempt spent 4.4 GPU-seconds of this allocation before
# dying on a config KeyError; that is in the ledger.
#
# target_seconds_per_arm=1800 is the M3-B plan this converts throughput
# for: 8 arms x 1800s = 4.0 GPU-hours. Arithmetic input only -- no part of
# M3-B is approved or started here.
import yaml

cfg_path = Path("configs/m3_profile.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["run"]["ticket_hours"] = 0.35
cfg["profile"]["target_seconds_per_arm"] = 1800
cfg["data"]["data_dir"] = str(DATA_DIR)
cfg_path.write_text(yaml.safe_dump(cfg))
print(cfg_path.read_text())

In [ ]:
# run_m3_profile.py enforces its own ticket and writes one ledger entry in
# a finally block whatever happens. It snapshots after every arm, so a
# deadline abort (exit code 2) still leaves everything measured on disk.
result = subprocess.run([
    sys.executable, "scripts/run_m3_profile.py",
    "--config", str(cfg_path),
])
print(f"profile exit code: {result.returncode}")
print(Path("budget/ledger.jsonl").read_text())

In [ ]:
import json
profile = json.load(open(cfg["output"]["path"]))
print(json.dumps({k: v for k, v in profile.items() if k != "config"}, indent=2))